# 第 8 章实验 · 推荐系统：协同过滤、矩阵分解、双塔、负采样与 CTR 排序

> 对应正文：[08-推荐系统](../docs/4-专业方向/08-推荐系统/README.md) ·
> [02 协同过滤与矩阵分解](../docs/4-专业方向/08-推荐系统/02-协同过滤与矩阵分解.md) ·
> [03 双塔模型与向量召回](../docs/4-专业方向/08-推荐系统/03-双塔模型与向量召回.md) ·
> [04 排序模型](../docs/4-专业方向/08-推荐系统/04-排序模型.md)

**环境**：全部实验只需 `numpy / matplotlib / scikit-learn`，**纯离线可跑**——
协同过滤、矩阵分解 SGD、双塔训练、负采样对比全部用 numpy 手写复刻，不依赖 torch。
数据全部由代码构造、随机种子固定——从上到下完整执行即复现。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

np.random.seed(0)   # 固定随机种子，全 notebook 结果可复现

# 中文字体：Windows 优先微软雅黑/黑体，Mac/Linux 自动回退到检测到的字体
_available = {f.name for f in font_manager.fontManager.ttflist}
_cjk = [f for f in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
                    "PingFang SC", "WenQuanYi Micro Hei"] if f in _available]
plt.rcParams["font.sans-serif"] = _cjk + plt.rcParams["font.sans-serif"]
plt.rcParams["axes.unicode_minus"] = False    # 让负号正常显示

def softmax_rows(m):
    """数值稳定的逐行 softmax"""
    e = np.exp(m - m.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

print("NumPy", np.__version__, "| 中文字体:", _cjk)

## 实验一：item 相似度推荐——正文 4×4 手算的代码复现

正文 02 页的例子：4 个用户 × 4 部电影的评分矩阵，只用**共同评过分的用户**算电影之间的
余弦相似度，再从"刚看完《流浪地球》"出发推 Top-2"看了又看"；
顺带把 user-based（找最像的人）也算一遍——同一个结论从两条路得到。

In [ ]:
R = np.array([[5, 1, 5, 1],
              [4, 2, 0, 1],
              [1, 5, 2, 5],
              [2, 5, 1, 4]], dtype=float)     # 0 = 没看过（正文原矩阵）
users4 = ["小红", "小刚", "小美", "小李"]
movies4 = ["流浪地球", "泰坦尼克号", "星际穿越", "罗曼假日"]

def item_cos(R, i, j):
    """电影 i 与 j 的余弦相似度：只用在两部电影上都评过分的用户（正文同款函数）"""
    mask = (R[:, i] > 0) & (R[:, j] > 0)
    a, b = R[mask, i], R[mask, j]
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("—— item-based：看了《流浪地球》的人还看谁 ——")
sims = {}
for j in range(4):
    if j != 0:
        sims[j] = item_cos(R, 0, j)
        print(f"sim_cos(流浪地球, {movies4[j]}) = {sims[j]:.3f}")
top2 = sorted(sims.items(), key=lambda kv: -kv[1])[:2]
print("看了又看 Top-2：", [movies4[j] for j, _ in top2])

print("\n—— user-based：小刚还没看《星际穿越》，要不要推给他？ ——")
usims = {}
for v in [0, 2, 3]:                            # 只用三人都评过的电影 A、B、D
    a = np.array([R[1, m] for m in [0, 1, 3]])
    b = np.array([R[v, m] for m in [0, 1, 3]])
    usims[v] = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f"sim_cos(小刚, {users4[v]}) = {usims[v]:.3f}")
best = max(usims, key=usims.get)
print(f"最像的邻居：{users4[best]}（相似度 {usims[best]:.2f}），"
      f"ta 给《星际穿越》打了 {R[best, 2]:.0f} 分 → 推荐给小刚")
# 预期输出：sim(流浪地球, 星际穿越)=0.967 最高 → Top-2 = [星际穿越, 泰坦尼克号]；
#           最像小刚的是小红（0.97），她给星际穿越打 5 分 → 推荐 C——两条路同答案

In [ ]:
S = np.eye(4)
for i in range(4):
    for j in range(4):
        if i != j:
            S[i, j] = item_cos(R, i, j)        # 电影×电影的完整相似度矩阵
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(S, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(4), movies4, rotation=30, ha="right")
ax.set_yticks(range(4), movies4)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title("item-based 协同过滤：电影 × 电影余弦相似度")
ax.set_xlabel("电影（列）"); ax.set_ylabel("电影（行）")
fig.colorbar(im, label="余弦相似度")
plt.tight_layout(); plt.show()

**观察**：sim(流浪地球, 星际穿越)=0.967 一骑绝尘——两部的受众高度重合；
user-based 那边最像小刚的是小红（0.97），她给《星际穿越》打了 5 分 → 推荐。
同一个结论从"物以类聚"和"人以群分"两条路独立得到——这就是协同过滤：
完全不看物品内容，只看"大家怎么用"。

## 实验二：矩阵分解 SGD——5×5 稀疏矩阵补空格

把评分矩阵拆成 用户矩阵 P (5×k) × 物品矩阵 Q (5×k)，只在**有评分的格子上**做 SGD：
误差 e = 真实分 − 预测分，两个隐向量互相朝对方拉（正则项往原点拉防过拟合）。
训练完看两件事：已评分格子的 RMSE 降到多少；**空格被补成了什么值**。

In [ ]:
np.random.seed(0)
R5 = np.array([[5, 1, 5, 1, 4],
               [4, 2, 0, 1, 5],
               [1, 5, 2, 5, 1],
               [2, 5, 1, 4, 0],
               [0, 4, 1, 5, 0]], dtype=float)   # 0 = 没看过：共 4 个空格
users5 = ["小红", "小刚", "小美", "小李", "小芳"]
movies5 = ["流浪地球", "泰坦尼克号", "星际穿越", "罗曼假日", "星球大战"]
observed = [(u, i) for u in range(5) for i in range(5) if R5[u, i] > 0]
print(f"5×5 矩阵共 {len(observed)} 个已评分格子、4 个空格")

k = 2                                          # 隐向量维度：玩具数据 2 维≈"科幻/爱情"
P = np.random.normal(0, 0.1, (5, k))           # 用户矩阵
Q = np.random.normal(0, 0.1, (5, k))           # 物品矩阵
lr, lam, epochs = 0.02, 0.02, 5000
rmse_before = np.sqrt(np.mean([(R5[u, i] - P[u] @ Q[i]) ** 2 for u, i in observed]))

for ep in range(epochs):                       # SGD：逐条评分更新（正文同款）
    for u, i in observed:
        e = R5[u, i] - P[u] @ Q[i]             # 误差 = 真实分 − 预测分
        Pu, Qi = P[u].copy(), Q[i].copy()
        P[u] += lr * (e * Qi - lam * Pu)       # p_u ← p_u + η(e·q_i − λp_u)
        Q[i] += lr * (e * Pu - lam * Qi)       # q_i ← q_i + η(e·p_u − λq_i)

rmse_after = np.sqrt(np.mean([(R5[u, i] - P[u] @ Q[i]) ** 2 for u, i in observed]))
print(f"训练前 RMSE = {rmse_before:.2f}（随机小向量 ≈ 全猜 0）")
print(f"训练后 RMSE = {rmse_after:.3f}（21 个观测把 2×(5+5) 个参数钉住了）\n")

pred = P @ Q.T
print("预测矩阵（括号 = 原空格）：")
print("        " + "  ".join(movies5))
for u in range(5):
    row = []
    for i in range(5):
        v = f"{pred[u, i]:4.1f}"
        row.append(f"({v.strip()})" if R5[u, i] == 0 else v)
    print(f"{users5[u]}  " + "  ".join(row))
for u, i in [(1, 2), (3, 4), (4, 0), (4, 4)]:   # 4 个空格的补全结果
    print(f"补全空格：{users5[u]} × {movies5[i]} = {pred[u, i]:.2f}")
# 预期输出：RMSE 3.50 → ~0.40；小刚 × 星际穿越 ≈ 4.9（科幻迷×科幻片，推荐！）；
#           小李/小芳 × 科幻片 ≈ 0.5~1.0（爱情迷不推科幻）——空格补得符合口味结构

**观察**：RMSE 从 3.50 降到 0.40 左右；最关键的空格——**小刚 × 星际穿越 ≈ 4.9 → 推荐**，
与实验一 user-based 的结论一致，而且这次完全没"借用"任何邻居，是坐标相乘算出来的。
泛化能力来自压缩：21 个观测钉住了 2×(5+5)−4 个自由度（正文深潜的自由度账），
模型被迫学出"科幻/爱情"结构，才能合理填充空格。

## 实验三：双塔最小实现（numpy）——两塔线性 + 内积 + softmax 训练

正文 03 页双塔的 numpy 版：8 个用户、8 部电影（4 科幻 + 4 爱情），每人 4 维合成特征
（科幻口味 / 爱情口味 / 两个真实口味维）。两塔都是**线性映射**，用户向量与物品向量
做内积、除以温度 τ、对全部 8 个物品做 softmax + 交叉熵——梯度还是第 7 章那条 **p − y**。
训练目标：每个用户的"点击"= 真实偏好 top-2；评估：Recall@3（真实 top-3 有几个进模型 top-3）。

In [ ]:
rng = np.random.default_rng(0)
n_users, n_items, feat, dim, tau = 8, 8, 4, 4, 0.5
item_names = ["流浪地球", "星际穿越", "三体", "沙丘",
              "泰坦尼克号", "罗曼假日", "恋恋笔记本", "爱在黎明"]

user_feat = np.zeros((n_users, feat))
user_feat[:4, 0] = 1.0                        # 用户 0~3：科幻迷
user_feat[4:, 1] = 1.0                        # 用户 4~7：爱情迷
user_feat[:, 2:] = rng.normal(0, 0.5, (n_users, 2))   # 两个真实口味维（如纪录片/喜剧）
item_feat = np.zeros((n_items, feat))
item_feat[:4, 0] = 1.0                        # 物品 0~3：科幻片
item_feat[4:, 1] = 1.0                        # 物品 4~7：爱情片
item_feat[:, 2:] = rng.normal(0, 0.5, (n_items, 2))

true_score = user_feat @ item_feat.T + rng.normal(0, 0.05, (n_users, n_items))
true_top3 = np.argsort(-true_score, axis=1)[:, :3]     # 评估 golden 集
clicked = np.argsort(-true_score, axis=1)[:, :2]       # 每人点击 = 真实 top-2
Y = (np.eye(n_items)[clicked[:, 0]] + np.eye(n_items)[clicked[:, 1]]) / 2   # 软目标各 0.5

Wu = rng.normal(0, 0.1, (feat, dim))          # 用户塔参数：4 维特征 → 4 维向量
Wi = rng.normal(0, 0.1, (feat, dim))          # 物品塔参数

def recall_at_3(Wu, Wi):
    s = (user_feat @ Wu) @ (item_feat @ Wi).T  # 上线检索：只需一次点积
    top3 = np.argsort(-s, axis=1)[:, :3]
    return np.mean([len(set(top3[u]) & set(true_top3[u])) / 3 for u in range(n_users)])

print("训练前（随机初始化）Recall@3 =", round(recall_at_3(Wu, Wi), 2))
lr, epochs, history = 0.5, 2000, []
for ep in range(epochs):
    Zu, Zi = user_feat @ Wu, item_feat @ Wi    # 两塔各自出向量（离线可预计算物品塔）
    P = softmax_rows(Zu @ Zi.T / tau)          # 内积 / 温度 → 全库 softmax
    loss = -(Y * np.log(P + 1e-12)).sum(axis=1).mean()
    history.append(loss)
    G = (P - Y) / n_users                      # softmax+交叉熵梯度 p − y（第 7 章同款）
    Wu -= lr * (user_feat.T @ (G @ Zi / tau))  # 反传到用户塔
    Wi -= lr * (item_feat.T @ (G.T @ Zu / tau))# 反传到物品塔
    if ep % 500 == 0:
        print(f"epoch {ep:4d}  loss = {loss:.4f}")
print(f"训练后 Recall@3 = {recall_at_3(Wu, Wi):.2f}（真实 top-3 几乎全被召回）")

s = (user_feat @ Wu) @ (item_feat @ Wi).T
print("\n用户 0（科幻迷）的检索排序：")
for i in np.argsort(-s[0]):
    tag = "真爱 top3" if i in true_top3[0] else "不爱"
    print(f"  {item_names[i]}  {s[0, i]:+6.2f}  {tag}")
# 预期输出：Recall@3 从 ~0.46 升到 ~0.92；用户 0 的排序前 4 名全是科幻片、
#           后 4 名全是爱情片——双塔学会了"按口味聚向量"

In [ ]:
plt.figure(figsize=(6.5, 3.5))
plt.plot(history)
plt.title("双塔训练：softmax 交叉熵损失（梯度 = p − y）")
plt.xlabel("训练轮次"); plt.ylabel("损失")
plt.tight_layout(); plt.show()
# 预期输出：loss 从 ~2.0（= ln 8，均匀猜）快速降到 ~0.9 后趋平——
#           剩余损失主要是"同类型 4 部片里分出 top-2"的细粒度排序

**观察**：Recall@3 从随机初始化的 ~0.46 升到 **~0.92**；用户 0 的检索结果前 4 全是科幻片。
双塔的全部机制就位：两塔独立出向量（物品塔可离线预计算建索引）→ 内积/温度 → softmax
训练（梯度 p−y，与第 7 章注意力/LLM 同一条）。真实系统把"线性塔"换成 DNN、
把全库 softmax 换成 in-batch negative（下个实验讲它的坑）。

## 实验四：负采样对比——随机均匀负 vs 热门偏置负

正文 03 页"流行度偏差"的实验版：20 个用户 × 30 个物品，头部物品**优质到人人都爱**
（热门 = 高质量，热是因为好），但**曝光 ∝ 流行度**——于是总有"用户真心喜欢、
只是还没曝光给他"的热门物品没被点击。正样本都是真实点击；负样本从未点击里采：
**方式 A** 均匀随机，**方式 B** 按流行度加权（热门更容易被采成负例）。
训练数据完全相同，只差负样本名单的分布——看谁的"隐藏热门真爱"召回率高。

In [ ]:
def run_once(seed, neg_mode, n_users=20, n_items=30, k_dim=2, steps=10000):
    rng = np.random.default_rng(seed)
    pop = 1.0 / np.arange(1, n_items + 1) ** 0.8    # 齐普夫式流行度：前几个是热门
    pop /= pop.sum()
    quality = 4.0 * pop / pop.max()                 # 头部优质到"人人都爱"（热是因为好）
    T = rng.normal(0, 1, (n_users, k_dim))          # 用户真实口味向量
    A = rng.normal(0, 1, (n_items, k_dim))          # 物品真实属性向量
    true_score = T @ A.T + quality[None, :] + rng.normal(0, 0.3, (n_users, n_items))
    liked = np.argsort(-true_score, axis=1)[:, :3]  # 每人真心喜欢 top-3
    exposed = rng.random((n_users, n_items)) < np.minimum(pop[None, :] * 4.0, 1.0)  # 曝光∝流行度
    clicked = np.zeros((n_users, n_items), dtype=bool)
    for u in range(n_users):
        for i in liked[u]:
            clicked[u, i] = exposed[u, i]           # 喜欢且被曝光 → 才会点击
    hidden = [(u, i) for u in range(n_users) for i in liked[u] if not exposed[u, i]]

    P = rng.normal(0, 0.1, (n_users, k_dim))        # 点积模型（与实验二同骨架）
    Q = rng.normal(0, 0.1, (n_items, k_dim))
    lr, lam, n_neg = 0.05, 0.001, 2                 # 每条正样本配 2 个负样本
    pos_pairs = [(u, i) for u in range(n_users) for i in range(n_items) if clicked[u, i]]
    for step in range(steps):
        u, i = pos_pairs[rng.integers(len(pos_pairs))]      # 采一条正样本（点击）
        pool = np.array([j for j in range(n_items) if not clicked[u, j]])
        pw = pop[pool] if neg_mode == "pop" else np.ones(len(pool))   # A/B 差异：负例权重
        js = rng.choice(pool, size=n_neg, replace=False, p=pw / pw.sum())
        pu, qi = P[u].copy(), Q[i].copy()
        e_pi = 1 / (1 + np.exp(-(pu @ qi)))         # 正样本的预测点击概率
        P[u] += lr * ((1 - e_pi) * qi - lam * pu)   # 拉近正样本
        Q[i] += lr * ((1 - e_pi) * pu - lam * qi)
        for j in js:                                # 推远负样本（B 模式热门被反复推）
            qj = Q[j].copy()
            e_pj = 1 / (1 + np.exp(-(pu @ qj)))
            P[u] -= lr * e_pj * qj
            Q[j] -= lr * e_pj * pu

    s = P @ Q.T
    top3 = np.argsort(-s, axis=1)[:, :3]
    rec_all = np.mean([len(set(top3[u]) & set(liked[u])) / 3 for u in range(n_users)])
    hot = set(range(3))                             # 最热门的 3 个物品
    hh = [(u, i) for u, i in hidden if i in hot]    # 隐藏的"热门真爱"（未曝光但真心喜欢）
    rec_hot = np.mean([i in top3[u] for u, i in hh])
    return rec_all, rec_hot, s[:, :3].mean()

print("负采样策略        全部Recall@3   隐藏热门召回   热门物品均分")
results = {}
for mode, label in [("uniform", "A 随机均匀负"), ("pop", "B 热门偏置负")]:
    ra, rh, hs = [], [], []
    for seed in range(5):                           # 5 个种子取平均，结论稳定
        a, h, sc = run_once(seed, mode)
        ra.append(a); rh.append(h); hs.append(sc)
    results[mode] = (np.mean(ra), np.mean(rh), np.mean(hs))
    print(f"{label}        {np.mean(ra):.2f}          {np.mean(rh):.2f}          {np.mean(hs):.2f}"
          f"   （隐藏热门召回各种子 {np.round(rh, 2).tolist()}）")
# 预期输出：正样本完全相同，只差负样本名单的分布——
#           隐藏热门召回 A≈0.52 vs B≈0.37；热门物品均分 A≈-0.3 vs B≈-0.9（被打压约 3 倍）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
labels = ["随机均匀负", "热门偏置负"]
vals1 = [results["uniform"][1], results["pop"][1]]
axes[0].bar(labels, vals1, color=["tab:blue", "tab:red"])
axes[0].set_title("隐藏热门真爱的召回率（5 种子平均）")
axes[0].set_ylabel("Recall@3"); axes[0].set_ylim(0, 0.65)
for x, v in zip(range(2), vals1):
    axes[0].text(x, v + 0.015, f"{v:.2f}", ha="center")
vals2 = [results["uniform"][2], results["pop"][2]]
axes[1].bar(labels, vals2, color=["tab:blue", "tab:red"])
axes[1].set_title("热门物品的平均模型得分（被打压程度）")
axes[1].set_ylabel("平均得分")
for x, v in zip(range(2), vals2):
    axes[1].text(x, v + (0.04 if v >= 0 else -0.09), f"{v:.2f}", ha="center")
plt.tight_layout(); plt.show()

**观察**：两种负采样的**正样本一模一样**，只因负样本名单的分布不同——
"隐藏的热门真爱"召回从 ~0.52 掉到 ~0.37，热门物品的平均得分被打压约 3 倍（-0.3 → -0.9）。
这就是正文说的流行度偏差：模型学到的不是"用户不喜欢它"，而是"它出镜太多被骂惨了"。
工业解法是 **logQ 纠偏**：给热门物品的负例 logit 减去 log q（出场费扣掉，让推远压力
与流行度脱钩），或 in-batch negative 必配纠偏——见文末改参数建议第 3 条。

## 实验五：CTR 排序玩具（sklearn）——LR + 交叉特征 + AUC

正文 04 页的玩具曝光日志：特征 = [用户偏好体育, 视频是体育, 视频是美食, 视频是数码, 晚间]，
外加一列**手工交叉特征**（体育用户 × 体育视频）。用逻辑回归学"会不会点"，
对比**有 / 无交叉特征**的 AUC——"体育迷看体育"这种组合信号，线性模型自己拼不出来。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = np.array([                                   # 玩具曝光日志（正文 04 页同款）
    [1,1,0,0,1,1],[1,1,0,0,0,1],[1,1,0,0,1,1],[1,1,0,0,0,1],[1,1,0,0,1,1],[1,1,0,0,0,1],
    [1,0,1,0,1,0],[1,0,0,1,0,0],[1,0,1,0,0,0],[1,0,0,1,1,0],[1,0,1,0,1,0],[1,0,0,1,0,0],
    [1,0,0,1,1,0],[1,0,0,1,0,0],[1,0,0,1,1,0],
    [0,0,1,0,1,0],[0,0,1,0,0,0],[0,0,1,0,1,0],[0,0,1,0,0,0],[0,0,1,0,1,0],[0,0,1,0,0,0],
    [0,1,0,0,1,0],[0,1,0,0,0,0],[0,1,0,0,1,0],[0,1,0,0,0,0],[0,1,0,0,1,0],[0,1,0,0,0,0],
])
y = np.array([1,1,1,1,1,1, 0,0,0,0,0,0, 1,0,1, 1,1,1,1,1,1, 0,0,0,0,0,0])   # 1=点击

m_full = LogisticRegression(max_iter=1000).fit(X, y)             # 含交叉特征
m_base = LogisticRegression(max_iter=1000).fit(X[:, :5], y)      # 去掉交叉特征
auc_full = roc_auc_score(y, m_full.predict_proba(X)[:, 1])
auc_base = roc_auc_score(y, m_base.predict_proba(X[:, :5])[:, 1])
print(f"AUC（含交叉特征）= {auc_full:.3f}")
print(f"AUC（无交叉特征）= {auc_base:.3f}   → 交叉特征值 {auc_full - auc_base:+.3f}")
print(f"交叉特征（体育×体育）权重 = {m_full.coef_[0][5]:.3f}\n")

cands = np.array([[1,1,0,0,1,1],                # 给"体育迷晚间刷 App"的三个候选打分
                  [1,0,0,1,1,0],
                  [1,0,1,0,1,0]])
probs = m_full.predict_proba(cands)[:, 1]
for name, p in zip(["体育视频", "数码视频", "美食视频"], probs):
    print(f"预测点击率 {name} = {p:.3f}")
# 预期输出：AUC 0.929 vs 0.731——交叉特征一个特征顶半边天；交叉权重 1.713；
#           体育 0.743 > 美食 0.597 > 数码 0.398（"体育迷也常点数码"的证据太稀疏，LR 没学进去）

**观察**：一列交叉特征把 AUC 从 0.731 抬到 0.929；"体育迷×体育视频"的权重高达 1.713。
注意数码被低估的细节——"体育迷也常点数码"的证据只有 3 行，线性模型学不动稀疏交叉，
这正是 FM/DeepFM 用隐向量共享参数要解的痛点（正文 04 页"深问 1"的现场版）。

## 实验六：温度对检索的影响——softmax(内积/τ) 的三种体温

正文 03 页"Gibbs 分布深潜"的可跑版：候选打分 s=[0.9, 0.7, 0.3]（归一化内积的典型量级），
看 τ=0.05 / 0.2 / 1 时 softmax(内积/τ) 的分布与熵。
τ→0⁺ 全押最大分（贪心/纯利用），τ→∞ 人人平分（纯探索）；
熵以 Var(s)/τ³ 的速率随温度上升——"探索油门"上刻着刻度。

In [ ]:
scores = np.array([0.9, 0.7, 0.3])
cands6 = ["候选1（分0.9）", "候选2（分0.7）", "候选3（分0.3）"]

def softmax_tau(s, tau):
    z = s / tau
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    return -np.sum(p * np.log(p + 1e-12))      # 熵：分布的"探索度"

print("τ      概率分布                熵 H     前两名几率比")
for tau in [0.05, 0.2, 1.0]:
    p = softmax_tau(scores, tau)
    print(f"{tau:.2f}   {np.round(p, 3).tolist()}    {entropy(p):.3f}    "
          f"{p[0]/p[1]:6.1f}:1")
print("理论几率比 = e^(Δs/τ)：Δs=0.2 →", [f"{np.exp(0.2/t):.1f}:1" for t in [0.05, 0.2, 1.0]])
# 预期输出：τ=0.05 → [0.98, 0.02, 0.00] 几乎 one-hot（贪心/纯利用），熵 ≈ 0.07
#           τ=0.2  → [0.71, 0.26, 0.03] 有主见地探索，熵 ≈ 0.81
#           τ=1.0  → [0.42, 0.35, 0.23] 接近均匀（乱撞/纯探索），熵 ≈ 1.07（上限 ln3≈1.10）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
w = 0.25                                        # 左图：三种 τ 的检索分布
for idx, (tau, c) in enumerate(zip([0.05, 0.2, 1.0], ["tab:red", "tab:orange", "tab:blue"])):
    p = softmax_tau(scores, tau)
    axes[0].bar(np.arange(3) + (idx - 1) * w, p, width=w, label=f"τ={tau}", color=c)
axes[0].set_xticks(range(3), cands6, fontsize=9)
axes[0].set_title("同一打分、三种体温下的检索分布")
axes[0].set_ylabel("概率"); axes[0].legend()

taus = np.linspace(0.02, 1.5, 100)              # 右图：熵随 τ 单调上升
axes[1].plot(taus, [entropy(softmax_tau(scores, t)) for t in taus])
for t in [0.05, 0.2, 1.0]:
    axes[1].scatter([t], [entropy(softmax_tau(scores, t))], zorder=3)
    axes[1].annotate(f"τ={t}", (t, entropy(softmax_tau(scores, t))),
                     textcoords="offset points", xytext=(6, -4))
axes[1].axhline(np.log(3), ls=":", color="gray", label="均匀分布熵上限 ln3")
axes[1].set_title("熵 H(τ) 随温度单调上升（dH/dτ = Var(s)/τ³ ≥ 0）")
axes[1].set_xlabel("温度 τ"); axes[1].set_ylabel("熵"); axes[1].legend()
plt.tight_layout(); plt.show()

**观察**：τ=0.05 时概率几乎全押在最高分上（确定性取 Top-K 的线上链路里 τ 不影响结果，
τ 起作用的是**训练损失**与**带随机性的探索分发**）；τ=1 时三家接近均分、熵逼近 ln 3。
这枚 τ 与第 7 章 LLM 的采样温度是同一个 Gibbs 分布：那头管"选哪个词"，这头管"选哪个物品"。

## 改参数建议（一次只改一个，观察一个）

1. **实验二**：k 从 2 改 16——训练 RMSE 可以更低，但空格预测开始"乱猜"（16 个参数拟合每个用户 4 条评分 = 背题）；再把 lam 改 0 和 1 对比 RMSE——正则太弱向量飘、太强向量萎缩；
2. **实验三**：τ 从 0.5 改 0.05，看收敛是否变慢、Recall 是否回落——低温把梯度集中在"还没分对"的样本上，没有难负例时训练空转（温度与难负例必须联调）；
3. **实验四**：把负采样从 `pop[pool]` 改回 `pop[pool] ** 0.75`（word2vec 的 3/4 次幂），看打压变温和多少；再给热门负例的 logit 减去 `log q`（logQ 纠偏），看隐藏热门召回能否恢复到 A 模式水平；
4. **实验五**：把"体育迷×数码"那 3 行样本复制成 30 行重训——数码的预测分会显著上升（交叉证据越稀疏 LR 越学不动）；或删掉交叉特征列看 AUC 掉多少；
5. **实验六**：τ 加一档 0.01，看分布完全 one-hot、熵趋近 0（纯贪心）；再把打分差距 Δs 从 0.2 改 0.02，看同样的 τ 下分布"体温"完全不同——τ 的刻度必须锁定在固定的分数标定下才有意义。